In [3]:
# !pip install -q scikit-learn pandas matplotlib seaborn

print("✅ Бібліотеки встановлено")

# =========================
# Блок 1. Імпорт та початкові налаштування
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # у цьому прикладі не обов'язково, просто імпорт

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

np.set_printoptions(edgeitems=3, linewidth=120)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

print("✅ Імпорти виконано.")

✅ Бібліотеки встановлено
✅ Імпорти виконано.


In [4]:
# =========================
# Блок 2. Завантаження даних
# =========================
print("\n=== Етап 1: Завантаження даних ===")
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("✅ Дані завантажено: Wisconsin Breast Cancer (scikit-learn)")
print(f"   Розмір ознак X: {X.shape} (рядків, ознак)")
print(f"   Розмір цілі y: {y.shape}")
print(f"   Назви ознак (всього {X.shape[1]}):", ", ".join(list(X.columns[:6])) + " ...")
print(f"   Мітки класів: {list(data.target_names)}  (УВАГА: у цьому наборі 0 = {data.target_names[0]}, 1 = {data.target_names[1]})")

class_counts = y.value_counts().sort_index()
print(f"   Баланс класів: 0 -> {class_counts[0]}, 1 -> {class_counts[1]} (усього {len(y)})")



=== Етап 1: Завантаження даних ===
✅ Дані завантажено: Wisconsin Breast Cancer (scikit-learn)
   Розмір ознак X: (569, 30) (рядків, ознак)
   Розмір цілі y: (569,)
   Назви ознак (всього 30): mean radius, mean texture, mean perimeter, mean area, mean smoothness, mean compactness ...
   Мітки класів: [np.str_('malignant'), np.str_('benign')]  (УВАГА: у цьому наборі 0 = malignant, 1 = benign)
   Баланс класів: 0 -> 212, 1 -> 357 (усього 569)


In [5]:
# =========================
# Блок 3. Швидкий EDA (огляд)
# =========================
print("\n=== Етап 2: Дослідницький аналіз даних (EDA) ===")
print("→ Перші 5 рядків X:")
print(X.head().to_string(index=False))

print("\n→ Інформація про типи та пропуски:")
X.info()

print("\n→ Описова статистика для 'mean radius' та 'mean area' (різні масштаби):")
cols_demo = ['mean radius', 'mean area']
print(X[cols_demo].describe().to_string())

print("\nКоментар: бачимо, що 'mean area' має значно більший масштаб, ніж 'mean radius'. "
      "Це підтверджує, чому масштабування важливе для LR та SVM.")



=== Етап 2: Дослідницький аналіз даних (EDA) ===
→ Перші 5 рядків X:
 mean radius  mean texture  mean perimeter  mean area  mean smoothness  mean compactness  mean concavity  mean concave points  mean symmetry  mean fractal dimension  radius error  texture error  perimeter error  area error  smoothness error  compactness error  concavity error  concave points error  symmetry error  fractal dimension error  worst radius  worst texture  worst perimeter  worst area  worst smoothness  worst compactness  worst concavity  worst concave points  worst symmetry  worst fractal dimension
       17.99         10.38          122.80     1001.0          0.11840           0.27760          0.3001              0.14710         0.2419                 0.07871        1.0950         0.9053            8.589      153.40          0.006399            0.04904          0.05373               0.01587         0.03003                 0.006193         25.38          17.33           184.60      2019.0            0.1622

In [6]:
# =========================
# Блок 4. Train/Test та масштабування
# =========================
print("\n=== Етап 3: Підготовка даних ===")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"✅ Розбивка train/test виконана (80/20). Train: {X_train.shape[0]} рядків, Test: {X_test.shape[0]} рядків.")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("✅ Масштабування ознак виконано ТІЛЬКИ на train (fit) і застосовано до test (transform).")



=== Етап 3: Підготовка даних ===
✅ Розбивка train/test виконана (80/20). Train: 455 рядків, Test: 114 рядків.
✅ Масштабування ознак виконано ТІЛЬКИ на train (fit) і застосовано до test (transform).


In [13]:
# =========================
# Блок 5. Навчання моделей
# =========================
print("\n=== Етап 4: Тренування моделей ===")

# Модель 1: Логістична регресія (тільки на масштабованих даних)
model_lr = LogisticRegression(max_iter=2000, random_state=42)
model_lr.fit(X_train_scaled, y_train)
print("✅ Навчено Logistic Regression (на масштабованих ознаках).")

# Модель 2: SVM - лінійне ядро (масштабовані дані)
model_svm_linear = SVC(kernel='linear')
model_svm_linear.fit(X_train_scaled, y_train)
print("✅ Навчено SVM (linear kernel, масштабовані ознаки).")

# Модель 2b: SVM - RBF ядро (масштабовані дані)
model_svm_rbf = SVC(kernel='rbf')
model_svm_rbf.fit(X_train_scaled, y_train)
print("✅ Навчено SVM (rbf kernel, масштабовані ознаки).")

# Модель 3: Random Forest (без масштабування)
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train, y_train)
print("✅ Навчено RandomForest (без масштабування — дерева нечутливі до масштабу).")



=== Етап 4: Тренування моделей ===
✅ Навчено Logistic Regression (на масштабованих ознаках).
✅ Навчено SVM (linear kernel, масштабовані ознаки).
✅ Навчено SVM (rbf kernel, масштабовані ознаки).
✅ Навчено RandomForest (без масштабування — дерева нечутливі до масштабу).


In [8]:
# =========================
# Блок 6. Прогнози
# =========================
print("\n=== Прогнози на тесті ===")
y_pred_lr        = model_lr.predict(X_test_scaled)
y_pred_svm_lin   = model_svm_linear.predict(X_test_scaled)
y_pred_svm_rbf   = model_svm_rbf.predict(X_test_scaled)
y_pred_rf        = model_rf.predict(X_test)

print("✅ Прогнози отримано для всіх моделей.")



=== Прогнози на тесті ===
✅ Прогнози отримано для всіх моделей.


In [14]:
# =========================
# Блок 7. Оцінка якості (метрики + матриці змішування)
# =========================
def evaluate(name, y_true, y_pred, pos_label=1):
    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, pos_label=pos_label)
    rec = recall_score(y_true, y_pred, pos_label=pos_label)
    f1  = f1_score(y_true, y_pred, pos_label=pos_label)
    cm  = confusion_matrix(y_true, y_pred)  # формат [[TN, FP],[FN, TP]] якщо pos_label=1
    print(f"\n=== {name} ===")
    print(f"Accuracy={acc:.3f} | Precision(клас {pos_label})={pre:.3f} | Recall(клас {pos_label})={rec:.3f} | F1={f1:.3f}")
    print("Confusion matrix (rows=true, cols=pred):\n", cm)
    print("Classification report:")
    print(classification_report(y_true, y_pred, target_names=data.target_names))
    return {"Model": name, "Accuracy": acc, "Precision": pre, "Recall": rec, "F1": f1}

print("\n=== Етап 5: Оцінка моделей ===")
print("Примітка: у цьому наборі 0 = malignant, 1 = benign. pos_label=1 означає метрики для класу 'benign'.")

rows = []
rows.append(evaluate("LogisticRegression", y_test, y_pred_lr,      pos_label=1))
rows.append(evaluate("SVM-linear",         y_test, y_pred_svm_lin, pos_label=1))
rows.append(evaluate("SVM-rbf",            y_test, y_pred_svm_rbf, pos_label=1))
rows.append(evaluate("RandomForest",       y_test, y_pred_rf,      pos_label=1))

metrics_table = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
print("\n=== Зведена таблиця метрик (сортування за F1) ===")
print(metrics_table.to_string(index=False))

best_row = metrics_table.iloc[0]
print(f"\n✅ Найкращий F1 показала модель: {best_row['Model']} (F1={best_row['F1']:.3f}, "
      f"Accuracy={best_row['Accuracy']:.3f}, Precision={best_row['Precision']:.3f}, Recall={best_row['Recall']:.3f})")



=== Етап 5: Оцінка моделей ===
Примітка: у цьому наборі 0 = malignant, 1 = benign. pos_label=1 означає метрики для класу 'benign'.

=== LogisticRegression ===
Accuracy=0.982 | Precision(клас 1)=0.986 | Recall(клас 1)=0.986 | F1=0.986
Confusion matrix (rows=true, cols=pred):
 [[41  1]
 [ 1 71]]
Classification report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


=== SVM-linear ===
Accuracy=0.974 | Precision(клас 1)=0.986 | Recall(клас 1)=0.972 | F1=0.979
Confusion matrix (rows=true, cols=pred):
 [[41  1]
 [ 2 70]]
Classification report:
              precision    recall  f1-score   support

   malignant       0.95      0.98      0.96        42
      benign       0.99      0.97      0.98        72

    accurac